<a href="https://colab.research.google.com/github/Abdurhman-elgrm/Hands-On_Machine_Learning/blob/main/CHAPTER%2015%20Transformers%20for%20Natural%20Language%20Processing%20and%20Chatbots/mistral_model.ipynbmistral_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

access_token = userdata.get('token-hf-read-mistral')

In [2]:
from huggingface_hub import login
login(access_token)


In [3]:
from transformers import AutoTokenizer , AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(
                                    model_id, device_map="auto", dtype="auto")



config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [9]:
def generate(model , tokenizer ,prompt, max_new_tokens=50 , **generate_kwargs):
    inputs = tokenizer(prompt , return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.eos_token_id,
                                 **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
prompt = "List some places I should visit in Paris."
generate(mistral7b, mistral7b_tokenizer, prompt)

'List some places I should visit in Paris.\n\nI’m going to Paris in a few weeks and I’m looking for some places to visit. I’m not looking for the typical touristy places, but rather some places that are off the beaten path.\n\nI’'

In [12]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"


In [17]:
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt,max_new_tokens=100)

answer = extended_text[len(full_prompt):].strip().split("\nMe: ")[0]

print(answer)

The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.


In [18]:
class BobTheChatbot:  # or ChatBob if you prefer
    def __init__(self, model, tokenizer, introduction=bob_introduction,max_answer_length=10_000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length
    def chat(self, prompt):
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context,
            max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context or
                    len(answer) >= self.max_answer_length): break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [19]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)

In [20]:
bob.chat("List some places I should visit in Paris.")

'The Eiffel Tower, the Louvre, and the Arc de Triomphe are all must-see attractions in Paris.'

In [21]:
bob.chat("Tell me more about the first place.")


'The Eiffel Tower is a wrought iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower.'